# **1. Data Collection**

1. Giới thiệu về data mình chọn, có thông tin gì (background, nội dung về gì)
2. Từ đó đưa ra big question - Note: ***Làm bài phân tích này cho ai?*** --> Phụ huynh
3. Overview dataset
4. Phân tích chi tiết (Portuguese) - story telling
   + Theo thứ tự (hành vi rút ra của học sinh từ overview)
   + Hành vi - Môi trường ở nhà - Yếu tố khác (Sắp xếp theo thứ tự hợp lý)
   + Những câu hỏi / phân tích phải liên quan tới big question
5. Tóm tắt những insight **có ý nghĩa nhất**
6. Dựa vào phần kết luận thì có thể đưa ra các recommendations gì cho người nghe
   + Rec phải liên quan tới insight (ko đc ý chung chung)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Chứa thông tin và điểm số của các học sinh học môn Toán (Mathematics).
df_mat = pd.read_csv('/kaggle/input/datasets/mrigaankjaswal/student-performance-in-mathematics-and-portuguese/student-mat.csv', sep=';')

#Chứa thông tin và điểm số của các học sinh học môn Tiếng Bồ Đào Nha (Portuguese).
df_por = pd.read_csv('/kaggle/input/datasets/mrigaankjaswal/student-performance-in-mathematics-and-portuguese/student-por.csv', sep=';')

In [ ]:
df = pd.concat([
    df_mat.assign(Subject='Math'),
    df_por.assign(Subject='Portuguese')
], ignore_index=True)

df.head()

In [ ]:
df_mat.shape

In [ ]:
df_por.shape

In [ ]:
df.shape

In [ ]:
#Chuyển các cột có binary yes/no sang 1/0

cols = ['schoolsup','famsup','paid','activities',
        'nursery','higher','internet','romantic']

df[cols] = df[cols].replace({'yes':1, 'no':0}).astype('int8')

In [ ]:
# Absence
df['Absence Level'] = pd.cut(
    df['absences'],
    bins=[-1, 5, 15, 30, 100],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Failures
df['Failure Level'] = pd.cut(
    df['failures'],
    bins=[-1, 0, 2, 4],
    labels=['None', 'Few', 'Many']
)

# Alcohol
# Walc - weekend alcohol consumption (numeric: from 1 - very low to 5 - very high)
# Dalc - workday alcohol consumption (numeric: from 1 - very low to 5 - very high)
df['Total Alcohol'] = df['Dalc'] + df['Walc']

df['Alcohol Level'] = pd.cut(
    df['Total Alcohol'],
    bins=[1, 3, 5, 7, 9, 10],
    labels=['Very low', 'Low', 'Medium', 'High', 'Very high']
)

# Score
def score_group(x):
    if x <= 10:
        return 'Low'
    elif x <= 15:
        return 'Average'
    else:
        return 'High'

df['Score Level'] = df['G3'].apply(score_group)

# Education
#Medu - mother's education (numeric: 0 - none, 1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
#Fedu - father's education (numeric: 0 - none, 1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
def categorize_edu(x):
    if x < 1:
        return 'None'
    elif x < 2:
        return 'Primary education'
    elif x < 3:
        return '5th to 9th grade'
    elif x < 4:
        return 'Secondary education'
    else:
        return 'Higher education'

edu_order = ['None', 'Primary education', '5th to 9th grade', 'Secondary education', 'Higher education']

In [ ]:
merge_cols = [
    'school', 'sex', 'age', 'address', 'famsize', 'Pstatus',
    'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian'
]

df_merged = pd.merge(
    df_mat,
    df_por,
    on = merge_cols,
    how = 'inner',
    suffixes = ('_mat', '_por')
)

df_merged.shape

G1 - Điểm học kỳ 1 / đầu kỳ (từ 0 đến 20)

G2 - Điểm học kỳ 2 / giữa kỳ sau (từ 0 đến 20)

G3 - Điểm cuối cùng (final) của môn học (từ 0 đến 20, mục tiêu đầu ra)

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
print(f'--> There are {df_mat.shape[0]} Math students')
print(f'--> There are {df_por.shape[0]} Portuguese students')

In [ ]:
print(f"Dataset has: {df.duplicated().sum()} duplicate rows")

In [ ]:
print(f"Dataset has: {df.isnull().sum().sum()} missing values")

In [ ]:
# Lấy các cột numerical
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Lấy các cột categorical
categorical_cols = df.select_dtypes(include=['object']).columns

# Đếm số lượng
print("Số biến numerical:", len(numerical_cols))
print("Số biến categorical:", len(categorical_cols))

# Hiển thị tên cột
print("\nNumerical columns:")
print(list(numerical_cols))

print("\nCategorical columns:")
print(list(categorical_cols))

# **2. Data Exploration**


In [ ]:
# Filter numeric columns
numeric_df = df.select_dtypes(include = [float, int])

# Compute correlation matrix
corr = numeric_df.corr()

# Plot heatmap
plt.figure(figsize = (14, 7))
sns.heatmap(corr, annot = True, cmap = 'coolwarm', linewidths = 0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
#Những feature ảnh hưởng tới điểm G3?

corr['G3'].sort_values(ascending=False)

**Demographics**

In [ ]:
# ===== Distribution of Final Grades (G3) =====

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ===== Overall Grade Distribution =====
axes[0].hist(df['G3'],
             bins=10,
             edgecolor='black')

axes[0].set_title('Overall Final Grade Distribution',
                  fontsize=16,
                  fontweight='bold')

axes[0].set_xlabel('Final Grade (G3)', fontsize=13)
axes[0].set_ylabel('Number of Students', fontsize=13)

axes[0].tick_params(axis='both', labelsize=11)

# ===== Distribution by Subject =====
subjects = df['Subject'].unique()

for sub in subjects:
    subset = df[df['Subject'] == sub]

    axes[1].hist(subset['G3'],
                 bins=10,
                 alpha=0.6,
                 label=sub,
                 edgecolor='black')

axes[1].set_title('Final Grade Distribution by Subject',
                  fontsize=16,
                  fontweight='bold')

axes[1].set_xlabel('Final Grade (G3)', fontsize=13)
axes[1].set_ylabel('Number of Students', fontsize=13)

axes[1].legend(fontsize=11)

axes[1].tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Có sự chênh lệch về giới tính không? - Tỷ lệ giữa nam và nữ

fig, axes = plt.subplots(1, 3, figsize=(15,5))

colors_list = [
    ['#66c2a5', '#fc8d62'],  # Overall
    ['#8da0cb', '#e78ac3'],  # Math
    ['#a6d854', '#ffd92f']   # Portuguese
]

gender_map = {'F': 'Female', 'M': 'Male'}

# ===== Overall =====
overall_counts = df['sex'].replace(gender_map).value_counts()

axes[0].pie(
    overall_counts,
    labels=overall_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors_list[0],
    textprops={'fontsize': 14}   # tăng font
)

axes[0].set_title('Overall', fontsize=16)

# ===== Theo từng Subject =====
subjects = df['Subject'].unique()

for i, sub in enumerate(subjects):
    subset = df[df['Subject'] == sub]

    gender_counts = subset['sex'].replace(gender_map).value_counts()

    axes[i+1].pie(
        gender_counts,
        labels=gender_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors_list[i+1],
        textprops={'fontsize': 14}   # tăng font
    )

    axes[i+1].set_title(sub, fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
#Độ tuổi tập trung ở khoảng nào? Có xuất hiện outlier hay không? 

plt.figure(figsize=(15, 6))

# ===== Histogram =====
plt.subplot(1, 2, 1)
sns.histplot(data=df, x='age', hue='Subject', bins=8, kde=True)
plt.title("Age Distribution by Subject")

# ===== Boxplot =====
plt.subplot(1, 2, 2)
ax = sns.boxplot(data=df, x='Subject', y='age')
plt.title("Age Distribution & Outliers")

counts = df['Subject'].value_counts()
percents = counts / counts.sum() * 100

subjects = df['Subject'].unique()

for i, sub in enumerate(subjects):
    median_val = df[df['Subject'] == sub]['age'].median()
    
    ax.text(i, median_val,
            f'{counts[sub]}\n({percents[sub]:.1f}%)',
            ha='center', va='center',
            color='white',
            fontsize=10,
            fontweight='bold')

plt.tight_layout()

In [ ]:
#Bao nhiêu học sinh sống ở urban vs rural?

print(f"--> Number of Math students live in: {df_mat['address'].value_counts()} \n")
print(f"--> Number of Portuguese students live in: {df_por['address'].value_counts()} \n")

#Học sinh phân bố ở urban và rural như nào?
# address - student's home address type ('U' - urban or 'R' - rural)

address_map = {
    'U' : 'Urban',
    'R' : 'Rural'
}

plt.figure(figsize=(10,5))
colors = ['#4CAF50', '#FF7043']

# ===== Math =====
plt.subplot(1,2,1)
addr_counts_mat = df_mat['address'].map(address_map).value_counts()

plt.pie(addr_counts_mat,
        labels=addr_counts_mat.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors)

plt.title("Math: Urban vs Rural")

# ===== Portuguese =====
plt.subplot(1,2,2)
addr_counts_por = df_por['address'].map(address_map).value_counts()

plt.pie(addr_counts_por,
        labels=addr_counts_por.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors)

plt.title("Portuguese: Urban vs Rural")

plt.tight_layout()

In [ ]:
#Học sinh sống ở urban và rural có ảnh hưởng đến điểm số không?

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='address', y='G3', hue='Subject')
plt.title("Urban vs Rural vs Final Grade by Subject")

In [ ]:
# ===============================Bỏ==========================
#Học sinh ở thành thị (U) vs nông thôn (R) ai điểm cao hơn? 


df['address_label'] = df['address'].map(address_map)

order = ['Urban', 'Rural']
hue_order = sorted(df['Subject'].unique()) 

plt.figure(figsize=(7,5))

ax = sns.boxplot(
    data=df, 
    x='address_label', 
    y='G3', 
    hue='Subject',
    order=order,
    hue_order=hue_order
)

group_counts = df.groupby(['address_label','Subject']).size().reset_index(name='count')
group_counts['percent'] = group_counts.groupby('address_label')['count'].transform(lambda x: x/x.sum()*100)

for i, addr in enumerate(order):
    for j, sub in enumerate(hue_order):
        subset = df[(df['address_label'] == addr) & (df['Subject'] == sub)]
        
        if len(subset) > 0:
            median_val = subset['G3'].median()
            count = len(subset)
            percent = group_counts[
                (group_counts['address_label']==addr) & 
                (group_counts['Subject']==sub)
            ]['percent'].values[0]
            
            x_pos = i - 0.2 + j * 0.4
            
            ax.text(x_pos, median_val,
                    f'{count}\n({percent:.1f}%)',
                    ha='center', va='center',
                    color='white',
                    fontsize=9,
                    fontweight='bold')

plt.title('G3 Score by Address (Urban vs Rural)')
plt.xlabel('Address')
plt.ylabel('Final Grade (G3)')

**Family**

In [ ]:
# ===============================Bỏ==========================

#Trình độ học vấn của phụ huynh tập trung ở mức nào?
#Medu - mother's education (numeric: 0 - none, 1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
#Fedu - father's education (numeric: 0 - none, 1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)

df['Avg parents edu'] = (df_por['Medu'] + df_por['Fedu']) / 2

df['Parents Edu Level'] = df['Avg parents edu'].apply(categorize_edu)

plt.figure(figsize=(15,5))
sns.countplot(data=df, x='Parents Edu Level', hue='Subject', order=edu_order)

plt.title("Parents' Education Level (Grouped)")
plt.xlabel("Education Level")
plt.ylabel("Count")

In [ ]:
# ===============================Bỏ==========================

#Học sinh có cha mẹ học cao vs thấp thì điểm khác nhau không?

counts = df['Parents Edu Level'].value_counts()
print(counts)

order = ['Low','Medium','High','Very High']

plt.figure(figsize = (15, 8))
sns.boxplot(data=df, x='Parents Edu Level', y='G3', order=edu_order)

for i, level in enumerate(order):
    if level in counts:
        median = df[df['Parents Edu Level'] == level]['G3'].median()
        
        plt.text(i, median, 
                 f'n={counts[level]}',
                 ha='center', va='center',
                 color='white',
                 fontsize=10, fontweight='bold')

plt.title("G3 vs Parents' Education Level")
plt.xlabel("Parents Education Level")
plt.ylabel("Final Grade (G3)")

**Study behavior**

In [ ]:
# ===============================Bỏ==========================

#Trường nào có nhiều học sinh hơn?
#school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)

school_map = {
    'GP' : 'Gabriel Pereira',
    'MS' : 'Mousinho da Silveira'
}

sns.countplot(x = df['school'].map(school_map), hue = df['Subject'])
plt.title('Number of Students by School')

In [ ]:
# ===============================Bỏ==========================

#Điểm số và trường có liên quan nhau hay không?

sns.boxplot(data=df, x='school', y='G3', hue='Subject')
plt.title('G3 Score by School and Subject')

df.groupby('school')['G3'].mean()

In [ ]:
# ===============================Bỏ==========================

#Thời gian học tập hằng tuần của mỗi môn như nào?
# studytime - weekly study time (1 : <2 hours, 2 : 2 to 5 hours, 3 : 5 to 10 hours, or 4 : >10 hours)

studytime_map = {
    1: '<2 hours',
    2: '2-5 hours',
    3: '5-10 hours',
    4: '>10 hours'
}

studytime_order = ['<2 hours','2-5 hours','5-10 hours','>10 hours']

plt.figure(figsize=(12,5))

colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']

# ===== Math =====
plt.subplot(1,2,1)
mat_counts = df_mat['studytime'].map(studytime_map).value_counts().reindex(studytime_order)

plt.pie(mat_counts,
        labels=mat_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors)

plt.title("Math: Weekly Study Time (%)")

# ===== Portuguese =====
plt.subplot(1,2,2)
por_counts = df_por['studytime'].map(studytime_map).value_counts().reindex(studytime_order)

plt.pie(por_counts,
        labels=por_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors)

plt.title("Portuguese: Weekly Study Time (%)")

plt.tight_layout()

In [ ]:
# ===============================Bỏ==========================

#Xu hướng uống rượu của học sinh vào ngày trong tuần và cuối tuần?
#Dalc - workday alcohol consumption (numeric: from 1 - very low to 5 - very high)
#Walc - weekend alcohol consumption (numeric: from 1 - very low to 5 - very high)

alcohol_map = {
    1: 'Very low',
    2: 'Low',
    3: 'Medium',
    4: 'High',
    5: 'Very high'
}
alcohol_order = ['Very low', 'Low', 'Medium', 'High', 'Very high']

# ===== Dalc (Workday) =====
dalc_df = df.copy()
dalc_df['Dalc_label'] = dalc_df['Dalc'].map(alcohol_map)

dalc_pct = dalc_df.groupby(['Subject','Dalc_label']).size().reset_index(name='count')
dalc_pct['percent'] = dalc_pct.groupby('Subject')['count'].transform(lambda x: x/x.sum()*100)

# ===== Walc (Weekend) =====
walc_df = df.copy()
walc_df['Walc_label'] = walc_df['Walc'].map(alcohol_map)

walc_pct = walc_df.groupby(['Subject','Walc_label']).size().reset_index(name='count')
walc_pct['percent'] = walc_pct.groupby('Subject')['count'].transform(lambda x: x/x.sum()*100)

# ===== Plot =====
plt.figure(figsize=(14,6))

# Workday
plt.subplot(1,2,1)
sns.barplot(data=dalc_pct, x='Dalc_label', y='percent', hue='Subject',
            order=alcohol_order)
plt.title('Workday Alcohol Consumption (%)')
plt.ylabel('Percentage (%)')
plt.xlabel('Alcohol Level')

for i in plt.gca().containers:
    plt.bar_label(i, fmt='%.1f%%')

# Weekend
plt.subplot(1,2,2)
sns.barplot(data=walc_pct, x='Walc_label', y='percent', hue='Subject',
            order=alcohol_order)
plt.title('Weekend Alcohol Consumption (%)')
plt.ylabel('Percentage (%)')
plt.xlabel('Alcohol Level')

for i in plt.gca().containers:
    plt.bar_label(i, fmt='%.1f%%')

plt.tight_layout()
plt.show()

**Lifestyle**

In [ ]:
# ===============================Bỏ==========================

#Uống rượu có ảnh hưởng đến điểm số không?

df['Walc_label'] = df['Walc'].map(alcohol_map)
order = ['Very low', 'Low', 'Medium', 'High', 'Very high']

plt.figure(figsize=(8,5))
ax = sns.boxplot(data=df, x='Walc_label', y='G3', order=order)

counts = df['Walc_label'].value_counts()

for i, level in enumerate(order):
    if level in counts:
        median_val = df[df['Walc_label'] == level]['G3'].median()
        
        ax.text(i,
                median_val,
                f'{counts[level]}',
                ha='center',
                va='center',
                color='white',
                fontsize=10,
                fontweight='bold')

plt.title("G3 vs Weekend Alcohol Consumption")
plt.xlabel("Alcohol Level")
plt.ylabel("Final Grade (G3)")

In [ ]:
# ===============================Bỏ==========================


df_compare = df[['Subject', 'Dalc', 'Walc']].melt(
    id_vars='Subject',
    var_name='Type',
    value_name='Level'
)

df_compare['Level_label'] = df_compare['Level'].map(alcohol_map)

pct_df = df_compare.groupby(['Type','Level_label']).size().reset_index(name='count')
pct_df['percent'] = pct_df.groupby('Type')['count'].transform(lambda x: x/x.sum()*100)

plt.figure(figsize=(10,6))
sns.barplot(data=pct_df, x='Level_label', y='percent', hue='Type',
            order=alcohol_order)

plt.title('Alcohol Consumption: Weekday vs Weekend (%)')
plt.xlabel('Alcohol Level')
plt.ylabel('Percentage (%)')

for container in plt.gca().containers:
    plt.bar_label(container, fmt='%.1f%%')


**Academic Performance**

In [ ]:
# ===============================Bỏ==========================

#Điểm phân bố như thế nào?

df_scores = df.melt(
    id_vars='Subject', 
    value_vars=['G1', 'G2', 'G3'],
    var_name='Exam', 
    value_name='Score'
)

plt.figure(figsize=(10,6))
sns.histplot(
    data=df_scores,
    x='Score',
    hue='Subject',  
    multiple='dodge', 
    kde=True
)
plt.title('Score Distribution by Subject (Math vs Portuguese)')

In [ ]:
#Tại sao có nhiều học sinh điểm 0?

print((df[['G1','G2','G3']] == 0).sum())

In [ ]:
df.groupby(df['G3'] == 0)[['absences','failures','studytime']].mean()

In [ ]:
#Học sinh nghỉ học ở môn nào nhiều hơn?

sns.histplot(x = df['absences'], hue = df['Subject'], kde = True)

In [ ]:
Q1_absences = df['absences'].quantile(0.25)
Q3_absences = df['absences'].quantile(0.75)
IQR_absences = Q3_absences - Q1_absences

upper_bound_absences = Q3_absences + 1.5 * IQR_absences

outliers_absences = df[df['absences'] > upper_bound_absences]
outliers_absences[['absences', 'G3']].sort_values(by='absences', ascending=False).head()

In [ ]:
#Kiểm tra: vắng nhiều có bị G3 = 0 không?

df[df['absences'] > upper_bound_absences]['G3'].describe()

In [ ]:
# ===============================Bỏ==========================

#Vắng nhiều có ảnh hưởng điểm số hay ko
#G3 - final grade (numeric: from 0 to 20, output target)
#absences - number of school absences (numeric: from 0 to 93)

order_absences = ['Low', 'Medium', 'High']

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    data=df, 
    x='Absence Level', 
    y='G3', 
    order=order_absences
)

counts = df['Absence Level'].value_counts().reindex(order_absences, fill_value=0)

for i, level in enumerate(order_absences):
    median_val = df[df['Absence Level'] == level]['G3'].median()
    
    ax.text(
        i, median_val,
        f'{counts[level]}',
        ha='center',
        color='white',
        fontsize=10,
        fontweight='bold'
    )

plt.title("G3 vs Absence Level")
plt.xlabel("Absence Level")
plt.ylabel("Final Grade (G3)")

In [ ]:
# ===============================Bỏ==========================

#Vắng ảnh hưởng khác nhau giữa Math vs Portuguese không

sns.boxplot(x='Absence Level', y='G3', hue='school', data=df)

In [ ]:
# ===============================Bỏ==========================

# Xem học sinh vắng nhiều có điểm thấp nhiều hơn không?

levels = df['Absence Level'].dropna().unique()

fig, axes = plt.subplots(1, len(levels), figsize=(5*len(levels),5))

for i, level in enumerate(levels):
    subset = df[df['Absence Level'] == level]
    
    counts = subset['Score Level'].value_counts(normalize=True) * 100
    
    axes[i].pie(
        counts,
        labels=counts.index,
        autopct='%1.1f%%',
        startangle=90
    )
    
    axes[i].set_title(f'Absence Level: {level}')

plt.suptitle('G3 Distribution by Absence Level')
plt.tight_layout()

In [ ]:
# ===============================Bỏ==========================

#Thời gian tự học (studytime) ảnh hưởng thế nào đến G3?

studytime_order = ['<2 hours', '2-5 hours', '5-10 hours', '>10 hours']
sns.boxplot(x = df['studytime'].map(studytime_map), y = df['G3'],
           order=studytime_order)
plt.title('G3 Score by Study time')
plt.xlabel('Study time')
plt.ylabel('Final Grade (G3)')

In [ ]:
# ===============================Bỏ==========================

#Muốn học cao có ảnh hưởng tới G3?
#higher - wants to take higher education (binary: yes or no)

higher_map = {
    1 : 'Wants Higher Education',
    0 : 'No Higher Education'
}

sns.boxplot(
    x=df['higher'].map(higher_map),
    y=df['G3']
)

plt.title('G3 Score by Higher Education Aspiration')
plt.xlabel('Higher Education Goal')
plt.ylabel('Final Grade (G3)')

# 3. Por subject analysis

In [ ]:
# Filter numeric columns
numeric_df_por = df_por.select_dtypes(include = [float, int])

# Compute correlation matrix
corr_por = numeric_df_por.corr()

# Plot heatmap
plt.figure(figsize = (14, 7))
sns.heatmap(corr_por, annot = True, cmap = 'coolwarm', linewidths = 0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ===== Correlation of G3 with Other Factors =====

# Filter numeric columns
numeric_df_por = df_por.select_dtypes(include=[float, int])

# Correlation riêng với G3
g3_corr = numeric_df_por.corr()[['G3']].sort_values(by='G3', ascending=False)

# Plot heatmap
plt.figure(figsize=(2,10))

sns.heatmap(
    g3_corr,
    annot=True,
    cmap='coolwarm',
    linewidths=0.5,
    fmt='.2f'
)

plt.title('Correlation Between G3 and Other Factors',
          fontsize=16,
          fontweight='bold')

plt.yticks(fontsize=11)
plt.xticks(fontsize=12)

plt.show()

In [ ]:
# =============================================================================
# G3 Distribution by Past Failures
# =============================================================================

failure_map = {
    0: '0 Failures',
    1: '1 Failure',
    2: '2 Failures',
    3: '3+ Failures'
}

failure_order = ['0 Failures', '1 Failure', '2 Failures', '3+ Failures']

df_por['Failure Level'] = (
    df_por['failures']
    .replace({4: 3})
    .map(failure_map)
)

count_failures = (
    df_por['Failure Level']
    .value_counts()
    .reindex(failure_order, fill_value=0)
)

plt.figure(figsize=(10,6))

ax = sns.boxplot(
    data=df_por,
    x='Failure Level',
    y='G3',
    order=failure_order,
    hue='Failure Level',
    palette='Reds',
    legend=False,
    width=0.6
)

for i, level in enumerate(failure_order):

    subset = df_por[df_por['Failure Level'] == level]['G3']

    if len(subset) > 0:

        q1 = subset.quantile(0.25)
        q3 = subset.quantile(0.75)
        center_box = (q1 + q3) / 2

        n = count_failures[level]

        ax.text(
            i,
            center_box,
            f'{n}',
            ha='center',
            va='center',
            fontsize=10,
            fontweight='bold',
            color='black'
        )

plt.title('G3 Distribution by Past Academic Failures',
          fontsize=14)

plt.xlabel('Number of Past Failures')
plt.ylabel('Final Grade (G3)')

plt.show()

In [ ]:
# Trường học có ảnh hưởng điểm số không?
#school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)

order_school = ['Gabriel Pereira', 'Mousinho da Silveira']

# Tạo cột label
df_por['school_label'] = df_por['school'].map(school_map)

# Đếm số lượng
count_school = df_por['school_label'].value_counts().reindex(order_school, fill_value=0)

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    data=df_por,
    x='school_label',
    y='G3',
    order=order_school
)

for i, level in enumerate(order_school):
    subset = df_por[df_por['school_label'] == level]['G3']
    
    if len(subset) > 0:
        median_val = subset.median()
        n = count_school[level]

        ax.text(
            i, median_val, 
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title('G3 Score by School')
plt.xlabel('School')
plt.ylabel('Final Grade (G3)')

Lý do tại sao điểm số trường GP cao hơn trường MS? GP có yếu tố gì khác với MS? parents edu, studytime ảnh hưởng đến điểm số ntn?

In [ ]:
# Nếu cả cha và mẹ đều có học vấn cao, điểm số có vượt trội ko?

df_por['Parents Edu Level'] = ((df_por['Medu'] + df_por['Fedu']) / 2).apply(categorize_edu)

count_edu_parents = df_por['Parents Edu Level'].value_counts()

plt.figure(figsize=(10,5))

ax = sns.boxplot(
    data=df_por, 
    x='Parents Edu Level', 
    y='G3', 
    order=edu_order
)

for i, level in enumerate(edu_order):
    if level in count_edu_parents:
        subset = df_por[df_por['Parents Edu Level'] == level]['G3']
        median_por = subset.median()
        n = count_edu_parents[level]

        plt.text(
            i, median_por,  
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title("G3 vs Parents' Education Level")
plt.xlabel("Parents Education Level")
plt.ylabel("Final Grade (G3)")


In [ ]:
# Tại sao học vấn phụ huynh cao thì con điểm cao? 
# Do Thời gian học (studytime) hay do Điều kiện cơ sở vật chất (internet)
# internet - Internet access at home (binary: yes or no)
# studytime - weekly study time (1 : <2 hours, 2 : 2 to 5 hours, 3 : 5 to 10 hours, or 4 : >10 hours)

# Internet label
df_por['Internet Label'] = df_por['internet'].map({
    'yes': 'Internet',
    'no': 'No Internet'
})

# Studytime label
df_por['Studytime Label'] = df_por['studytime'].map(studytime_map)

In [ ]:
# ===============================Bỏ==========================

# Cha mẹ học cao → con có học nhiều hơn không?

pivot_study = df_por.pivot_table(
    index='Parents Edu Level',
    columns='Studytime Label',
    values='G3',
    aggfunc='mean'
)

plt.figure(figsize=(8,5))
sns.heatmap(pivot_study, annot=True, cmap='Blues', fmt=".1f")

plt.title("G3 by Parents Education & Study Time")
plt.xlabel("Study Time")
plt.ylabel("Parents Education Level")

In [ ]:
# ===============================Bỏ==========================

# Parents Edu + Internet

df_por['Avg parents edu'] = (df_por['Medu'] + df_por['Fedu']) / 2
df_por['Edu Group'] = df_por['Avg parents edu'].apply(lambda x: 'High Edu (>=3)' if x >= 3 else 'Low Edu (<3)')

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=df_por,
    x='Edu Group',
    y='G3',
    hue='Internet Label',
    palette='muted',
    capsize=.1,
    errorbar=None
)

plt.title("Average G3 Score by Parental Education Group and Internet Access")
plt.xlabel("Parental Education Group")
plt.ylabel("Average Final Grade (G3)")

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.show()

In [ ]:
# Có phải do điều kiện (internet)?

edu_order = [
    'None',
    'Primary education',
    '5th to 9th grade',
    'Secondary education',
    'Higher education'
]

df_por['Parents Edu Level'] = pd.Categorical(
    df_por['Parents Edu Level'],
    categories=edu_order,
    ordered=True
)

pivot_internet = df_por.pivot_table(
    index='Parents Edu Level',
    columns='Internet Label',
    values='G3',
    aggfunc='mean'
)

plt.figure(figsize=(6,5))
sns.heatmap(pivot_internet, annot=True, cmap='Greens', fmt=".1f")

plt.title("G3 by Parents Education & Internet Access")
plt.xlabel("Internet Access")
plt.ylabel("Parents Education Level")

plt.show()

In [ ]:
# =============================================================================
# Parents Education + Study Time vs G3
# =============================================================================

def regroup_edu(x):
    if x in ['None', 'Primary education']:
        return 'None/Primary'
    return x

df_por['Parents Edu Group'] = df_por['Parents Edu Level'].apply(regroup_edu)

new_edu_order = [
    'None/Primary',
    '5th to 9th grade',
    'Secondary education',
    'Higher education'
]

studytime_order = [
    '<2 hours',
    '2-5 hours',
    '5-10 hours',
    '>10 hours'
]

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=df_por,
    x='Parents Edu Group',
    y='G3',
    hue='Studytime Label',
    order=new_edu_order,
    hue_order=studytime_order,   # <-- add this
    palette='viridis',
    errorbar=None
)

# =============================================================================
# Styling
# =============================================================================

plt.title(
    "Average G3 Score by Parents' Education and Study Time",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Parents' Education Level")
plt.ylabel("Average Final Grade (G3)")

plt.legend(
    title='Study Time',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

# Add values on bars
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.2f',
        padding=3,
        fontsize=8
    )

plt.tight_layout()
plt.show()

Khi làm slide thêm 2light insight đặc biệt chú ý / khác biệt (in đậm / khoanh đỏ)

In [ ]:
#Những học sinh có gia đình êm ấm (famrel = 4, 5) có điểm số ổn định hơn không?
#famrel - quality of family relationships (numeric: from 1 - very bad to 5 - excellent)

famrel_map = {
    1: 'Very bad',
    2: 'Bad',
    3: 'Average',
    4: 'Good',
    5: 'Excellent'
}

order_famrel = ['Very bad', 'Bad', 'Average', 'Good', 'Excellent']

count_famrel = df_por['famrel'].map(famrel_map).value_counts().reindex(order_famrel, fill_value=0)

plt.figure(figsize=(7,5))

ax = sns.boxplot(
    x=df_por['famrel'].map(famrel_map),
    y=df_por['G3'],
    order=order_famrel
)

for i, level in enumerate(order_famrel):
    subset = df_por[df_por['famrel'].map(famrel_map) == level]['G3']
    
    if len(subset) > 0:
        median_val = subset.median()
        n = count_famrel[level]

        ax.text(
            i, median_val,   
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title("G3 Distribution by Family Relationship Quality")
plt.xlabel("Family Relationship")
plt.ylabel("Final Grade (G3)")

In [ ]:
# ===============================Bỏ==========================

#Việc đi chơi bao nhiêu là đủ và bao nhiêu là quá nhiều dẫn đến tụt điểm?

goout_map = {
    1: 'Very Low',
    2: 'Low',
    3: 'Medium',
    4: 'High',
    5: 'Very High'
}

order_goout = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

count_goout = (df_por['goout'].map(goout_map).value_counts().reindex(order_goout, fill_value=0))

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    x = df_por['goout'].map(goout_map),
    y = df_por['G3'],
    order = order_goout
)

for i, level in enumerate(order_goout):
    subset = df_por[df_por['goout'].map(goout_map) == level]['G3']
    
    if len(subset) > 0:
        median_val = subset.median()
        n = count_goout[level]

        ax.text(
            i,
            median_val + 0.5,  
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title("G3 vs Going Out Frequency")
plt.xlabel("Going Out Level")
plt.ylabel("Final Grade (G3)")


In [ ]:
#Uống rượu có ảnh hưởng đến điểm số không? 

df_por['Total Alcohol'] = df_por['Dalc'] + df_por['Walc']

df_por['Alcohol Level'] = pd.cut(
    df_por['Total Alcohol'],
    bins=[-1, 3, 5, 7, 9, 10],
    labels=['Very low', 'Low', 'Medium', 'High', 'Very high']
)

alcohol_order = ['Very low', 'Low', 'Medium', 'High', 'Very high']

count_alcohol = (
    df_por['Alcohol Level']
    .value_counts()
    .reindex(alcohol_order, fill_value=0)
)

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    data=df_por,
    x='Alcohol Level',
    y='G3',
    order=alcohol_order
)

for i, level in enumerate(alcohol_order):
    subset = df_por[df_por['Alcohol Level'] == level]['G3']
    
    if len(subset) > 0:
        median_val = subset.median()
        n = count_alcohol[level]

        ax.text(
            i, median_val - 0.8,
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title("G3 vs Alcohol Consumption")
plt.xlabel("Alcohol Level")
plt.ylabel("Final Grade (G3)")

In [ ]:
# Đi chơi nhiều có tệ bằng việc uống rượu nhiều không?
# Bản đồ nhiệt minh họa mối quan hệ giữa goout và việc tiêu thụ rượu vào cuối tuần liên quan đến điểm số cuối cùng của họ (G3).

pivot_cat = df_por.pivot_table(
    index='goout',
    columns='Alcohol Level',
    values='G3',
    aggfunc='mean',
    observed=False
)

pivot_cat.index = pivot_cat.index.map(goout_map)

plt.figure(figsize=(9, 5))

sns.heatmap(
    pivot_cat,
    annot=True,
    cmap='RdYlGn',
    fmt=".1f"
)

plt.title('Average G3 by Going Out and Alcohol Level')
plt.xlabel('Alcohol Consumption Level')
plt.ylabel('Going Out Frequency')

# Insight lạ

In [ ]:
# ===============================Bỏ==========================

# Vắng nhiều có ảnh hưởng điểm số hay ko
# G3 - final grade (numeric: from 0 to 20, output target)
# absences - number of school absences (numeric: from 0 to 93)

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df_por, 
    x='absences', 
    y='G3')


plt.title("G3 vs Absence Level")
plt.xlabel("Absence")
plt.ylabel("Final Grade (G3)")

In [ ]:
# ===============================Bỏ==========================

# Absence cho hs hoc nhieu / hoc it 

# Studytime label
df_por['Study Time'] = df_por['studytime'].map(studytime_map)

studytime_order = ['<2 hours', '2-5 hours', '5-10 hours', '>10 hours']

# Facet scatterplot
g = sns.FacetGrid(
    df_por,
    col='Study Time',
    col_order=studytime_order,
    height=4,
    aspect=1
)

g.map_dataframe(
    sns.scatterplot,
    x='absences',
    y='G3',
    alpha=0.6
)

g.set_axis_labels("Absences", "Final Grade (G3)")
g.set_titles("Study Time: {col_name}")

plt.subplots_adjust(top=0.85)
g.fig.suptitle("Relationship Between Absences and G3 by Study Time")

plt.show()

In [ ]:
# ===============================Bỏ==========================

studytime_order = ['<2 hours', '2-5 hours', '5-10 hours', '>10 hours']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, level in enumerate(studytime_order):
    ax = axes[i // 2, i % 2]
    
    subset = df_por[df_por['studytime'].map(studytime_map) == level]
    
    sns.scatterplot(
        data=subset,
        x='absences',
        y='G3',
        ax=ax
    )
    
    ax.set_title(f'Study Time: {level}')
    ax.set_xlabel('Absences')
    ax.set_ylabel('Final Grade (G3)')

plt.suptitle('G3 vs Absences by Study Time', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Có người yêu có làm học sinh trốn học nhiều hơn ko?
# romantic - with a romantic relationship (binary: yes or no)

plt.figure(figsize=(8,5))

sns.boxplot(
    data=df_por,
    x=df_por['romantic'].map({'no' : 'Single', 'yes' : 'In relationship'}),
    y='G3',
    hue=pd.cut(df_por['absences'], bins=[-1,5,15,30,100],
               labels=['Low absence', 'Medium absence', 'High absence', 'Very high absence'])
)

plt.title("G3 by Romantic Relationship and Absences")
plt.xlabel("Romantic Relationship")
plt.ylabel("Final Grade (G3)")
plt.legend(title='Absence Level')

#Đáng suy ngẫm

In [ ]:
# ===============================Bỏ==========================

# Có người yêu hệ quả lên điểm số như thế nào?

# Romantic label
df_por['Relationship Status'] = df_por['romantic'].map({
    'yes': 'In Relationship',
    'no': 'Single'
})

# Studytime order
studytime_order = ['<2 hours', '2-5 hours', '5-10 hours', '>10 hours']

plt.figure(figsize=(10,6))

sns.boxplot(
    data=df_por,
    x='Relationship Status',
    y='G3',
    hue=df_por['studytime'].map(studytime_map),
    hue_order=studytime_order,
    palette='Set2'
)

plt.title('Final Grade (G3) by Relationship Status and Study Time')
plt.xlabel('Relationship Status')
plt.ylabel('Final Grade (G3)')

plt.legend(
    title='Study Time',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()

In [ ]:
#===========================================BỎ============================================

# Romantic → Absences → G3

sns.scatterplot(
    data=df_por,
    x='absences',
    y='G3',
    hue='romantic'
)

plt.title("Absences vs G3 (Colored by Romantic Relationship)")

In [ ]:
# ===============================Bỏ==========================

# Một học sinh nhận được càng nhiều sự hỗ trợ thì điểm có cao hơn không?
# schoolsup + famsup + paid
# schoolsup - extra educational support (binary: yes or no)
# famsup - family educational support (binary: yes or no)
# paid - extra paid classes within the course subject (Math or Portuguese) (binary: yes or no)

support_cols = ['schoolsup', 'famsup', 'paid']

df_por['Support'] = (
    df_por[support_cols]
    .replace({'yes': 1, 'no': 0})
    .sum(axis=1)
)

count_support = df_por['Support'].value_counts().sort_index()

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    data=df_por,
    x='Support',
    y='G3'
)

for i, level in enumerate(sorted(df_por['Support'].unique())):
    subset = df_por[df_por['Support'] == level]['G3']
    if len(subset) > 0:
        median_val = subset.median()
        n = count_support[level]

        ax.text(
            i,
            median_val,
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color = 'white'
        )

plt.title("G3 by Educational Support")
plt.xlabel("Number of Supports")
plt.ylabel("Final Grade (G3)")

In [ ]:
#Thời gian tự học (studytime) ảnh hưởng thế nào đến G3?
# studytime - weekly study time (1 : <2 hours, 2 : 2 to 5 hours, 3 : 5 to 10 hours, or 4 : >10 hours)

studytime_map = {
    1: '<2 hours',
    2: '2-5 hours',
    3: '5-10 hours',
    4: '>10 hours'
}
studytime_order = ['<2 hours', '2-5 hours', '5-10 hours', '>10 hours']

count_studytime = df_por['studytime'].map(studytime_map).value_counts().reindex(studytime_order, fill_value=0)

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    x = df_por['studytime'].map(studytime_map),
    y = df_por['G3'],
    order = studytime_order
)

for i, level in enumerate(studytime_order):
    subset = df_por[df_por['studytime'].map(studytime_map) == level]['G3']
    
    if len(subset) > 0:
        median_val = subset.median()
        n = count_studytime[level]

        ax.text(
            i, median_val + 0.5, 
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title('G3 Score by Study Time')
plt.xlabel('Study Time')
plt.ylabel('Final Grade (G3)')

In [ ]:
#Muốn học cao có ảnh hưởng tới G3?
#higher - wants to take higher education (binary: yes or no)

higher_map = {
    'yes': 'Wants Higher Education',
    'no': 'No Higher Education'
}

order_higher = ['No Higher Education', 'Wants Higher Education']

count_higher = (
    df_por['higher'].map(higher_map)
    .value_counts()
    .reindex(order_higher, fill_value=0)
)

plt.figure(figsize=(8,5))

ax = sns.boxplot(
    x = df_por['higher'].map(higher_map),
    y = df_por['G3'],
    order=order_higher
)

for i, level in enumerate(order_higher):
    subset = df_por[df_por['higher'].map(higher_map) == level]['G3']
    
    if len(subset) > 0:
        median_val = subset.median()
        n = count_higher[level]

        ax.text(
            i,
            median_val - 1,   
            f'{n}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color='white'
        )

plt.title('G3 Score by Higher Education Aspiration')
plt.xlabel('Higher Education Goal')
plt.ylabel('Final Grade (G3)')

In [ ]:
# ===============================Bỏ==========================

# Sự thay đổi điểm số qua các kỳ.
# higher - wants to take higher education (binary: yes or no)

df_trend = df_por.melt(
    id_vars=['higher'],
    value_vars=['G1', 'G2', 'G3'],
    var_name='Period',
    value_name='Grade'
)

df_trend['Higher Education Goal'] = df_trend['higher'].map({
    'yes': 'Wants Higher Education',
    'no': 'No Higher Education Plan'
})

period_order = ['G1', 'G2', 'G3']

plt.figure(figsize=(10,6))

sns.lineplot(
    data=df_trend,
    x='Period',
    y='Grade',
    hue='Higher Education Goal',
    marker='o'
)

plt.title('Grade Progression Across Periods (G1 → G3)')
plt.xlabel('Academic Period')
plt.ylabel('Average Grade')

plt.grid(alpha=0.3)
plt.legend(title='Student Goal')

plt.show()

In [ ]:
#Học sinh ở nông thôn thường phải đi học xa hơn. 
#Hai yếu tố xung quanh này có vắt kiệt sức lực và làm giảm điểm số của họ?
#address - student's home address type (binary: "U" - urban or "R" - rural)
#traveltime - home to school travel time (numeric: 1 - <15 min., 2 - 15 to 30 min., 3 - 30 min. to 1 hour, or 4 - >1 hour)

df_por['Address Type'] = df_por['address'].map({'U': 'Urban', 'R': 'Rural'})

traveltime_map = {
    1: '<15 min',
    2: '15-30 min',
    3: '30-60 min',
    4: '>1 hour'
}

travel_order = ['<15 min', '15-30 min', '30-60 min', '>1 hour']

df_por['Travel Time'] = df_por['traveltime'].map(traveltime_map)

pivot_geo = df_por.pivot_table(
    index='Address Type',
    columns='Travel Time',
    values='G3',
    aggfunc='mean'
)

pivot_geo = pivot_geo[travel_order]

plt.figure(figsize=(10,5))

sns.heatmap(
    pivot_geo,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn'
)

plt.title('Average G3 by Address Type and Travel Time')
plt.xlabel('Travel Time to School')
plt.ylabel('Home Address Type')

plt.show()